# 01 Feature Engineering and Point in Time

**Purpose:** Demonstrate point‑in‑time feature engineering to avoid label leakage, materialize offline features to Delta, and prepare an online lookup key.

**Run order:** Execute cells sequentially on a Databricks cluster (Databricks Runtime for Machine Learning recommended). This notebook uses synthetic data so you can run it end‑to‑end without external datasets.

## 1 Install / verify dependencies

Run this cell if you haven't already installed the repo dependencies in this cluster session. The environment setup notebook installs these, but running here is safe and idempotent.

In [ ]:
%pip install -q delta-spark pandas scikit-learn mlflow

import importlib
for pkg in ["pandas","sklearn","delta","mlflow"]:
    try:
        m = importlib.import_module(pkg)
        print(pkg, getattr(m, "__version__", "unknown"))
    except Exception as e:
        print(pkg, "not available:", e)

## 2 Create synthetic event and label data

We generate two Delta tables:
- `demo.events` — time series of features per `entity_id` with `feature_ts` timestamps.
- `demo.labels` — label events with `label_ts` timestamps. Labels occur at specific times and must only use features available at or before `label_ts`.

This synthetic dataset is small and intended for lab purposes.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, TimestampType, DoubleType
import datetime, random

spark = SparkSession.builder.getOrCreate()

# Parameters
NUM_ENTITIES = 50
DAYS = 30
BASE = datetime.datetime(2023, 1, 1)

# Generate events
events = []
for entity in range(1, NUM_ENTITIES + 1):
    for day in range(0, DAYS):
        # random hour to add variety
        ts = BASE + datetime.timedelta(days=day, hours=random.randint(0, 23))
        f1 = round(random.random() * 10, 4)
        f2 = round(random.random() * 5, 4)
        events.append((entity, ts, float(f1), float(f2)))

events_schema = StructType([
    StructField("entity_id", IntegerType(), False),
    StructField("feature_ts", TimestampType(), False),
    StructField("f1", DoubleType(), False),
    StructField("f2", DoubleType(), False)
])

events_df = spark.createDataFrame(events, schema=events_schema)
events_df = events_df.repartition("entity_id")
events_df.write.format("delta").mode("overwrite").saveAsTable("demo.events")
print("Wrote demo.events with rows:", events_df.count())

# Generate labels (some days later)
labels = []
label_days = [15, 20, 25]
for entity in range(1, NUM_ENTITIES + 1):
    for day in label_days:
        ts = BASE + datetime.timedelta(days=day, hours=12)
        # binary label for simplicity
        label = float(random.choice([0, 1]))
        labels.append((entity, ts, label))

labels_schema = StructType([
    StructField("entity_id", IntegerType(), False),
    StructField("label_ts", TimestampType(), False),
    StructField("label", DoubleType(), False)
])

labels_df = spark.createDataFrame(labels, schema=labels_schema)
labels_df.write.format("delta").mode("overwrite").saveAsTable("demo.labels")
print("Wrote demo.labels with rows:", labels_df.count())

## 3 Inspect the data
Quick preview of events and labels to understand distribution and timestamps.

In [ ]:
display(spark.table("demo.events").orderBy(F.col("entity_id"), F.col("feature_ts")).limit(20))
display(spark.table("demo.labels").orderBy(F.col("entity_id"), F.col("label_ts")).limit(20))

## 4 Point‑in‑time join (offline training set assembly)

For each label row, find the latest event for the same `entity_id` where `feature_ts <= label_ts`. Use a window and `row_number()` to pick the most recent event per label.

In [ ]:
from pyspark.sql.window import Window

events = spark.table("demo.events").withColumn("feature_ts_unix", F.col("feature_ts").cast("long"))
labels = spark.table("demo.labels").withColumn("label_ts_unix", F.col("label_ts").cast("long"))

# Join labels to events where event time <= label time
joined = labels.alias("l").join(
    events.alias("e"),
    (F.col("l.entity_id") == F.col("e.entity_id")) & (F.col("e.feature_ts_unix") <= F.col("l.label_ts_unix")),
    how="left"
)

# Use row_number to pick the latest event per label
w = Window.partitionBy("l.entity_id", "l.label_ts_unix").orderBy(F.col("e.feature_ts_unix").desc())
joined_ranked = joined.withColumn("rn", F.row_number().over(w)).filter(F.col("rn") == 1)

training_set = joined_ranked.select(
    F.col("l.entity_id").alias("entity_id"),
    F.col("l.label_ts").alias("label_ts"),
    F.col("l.label").alias("label"),
    F.col("e.f1").alias("f1"),
    F.col("e.f2").alias("f2"),
    F.col("e.feature_ts").alias("feature_ts")
)

training_set.write.format("delta").mode("overwrite").saveAsTable("demo.training_set")
print("Wrote demo.training_set with rows:", training_set.count())
display(spark.table("demo.training_set").limit(20))

## 5 Validate no label leakage
Run assertions to ensure no `feature_ts` is after `label_ts` and that the training set size is reasonable (<= labels count).

In [ ]:
# Test 1: No feature_ts > label_ts
bad_count = spark.sql("SELECT COUNT(*) as cnt FROM demo.training_set WHERE feature_ts > label_ts").collect()[0]["cnt"]
assert bad_count == 0, f"Label leakage detected: {bad_count} rows have feature_ts > label_ts"

# Test 2: Training set rows <= labels rows
labels_count = spark.table("demo.labels").count()
training_count = spark.table("demo.training_set").count()
assert training_count <= labels_count, f"Training set ({training_count}) should not exceed labels ({labels_count})"
print("Validation passed: no label leakage and training set size OK")

## 6 Materialize aggregated features (example offline feature table)

Create a feature table that stores the latest timestamp and aggregated statistics per `entity_id`. This table can be used for online lookups or as a feature store source.

In [ ]:
features_to_materialize = spark.table("demo.events").groupBy("entity_id").agg(
    F.max("feature_ts").alias("last_feature_ts"),
    F.avg("f1").alias("avg_f1"),
    F.avg("f2").alias("avg_f2"),
    F.count("f1").alias("num_events")
)
features_to_materialize.write.format("delta").mode("overwrite").saveAsTable("demo.feature_table")
print("Wrote demo.feature_table with rows:", features_to_materialize.count())
display(spark.table("demo.feature_table").limit(20))

## 7 Example: Join materialized features to training set (enrich training rows)

This demonstrates how to enrich the assembled training set with the materialized features (e.g., last known aggregates). Note: ensure you join on keys and consider the timestamp semantics in production.

In [ ]:
training = spark.table("demo.training_set")
features = spark.table("demo.feature_table")

enriched = training.join(features, on="entity_id", how="left").select(
    "entity_id", "label_ts", "label", "f1", "f2", "feature_ts", "last_feature_ts", "avg_f1", "avg_f2", "num_events"
)
display(enriched.limit(20))
enriched.write.format("delta").mode("overwrite").saveAsTable("demo.enriched_training_set")
print("Wrote demo.enriched_training_set with rows:", enriched.count())

## 8 Notes and production considerations

- **Point‑in‑time correctness:** Always ensure the join condition strictly enforces `feature_ts <= label_ts` to avoid leakage.
- **Late arriving data:** Decide how to handle late events (backfills, reprocessing). Consider watermarking or incremental feature materialization.
- **Feature Store:** For production, register features in a Feature Store and materialize incremental updates for online serving.
- **Scaling:** Partition feature tables by `entity_id` or time ranges; use incremental pipelines for large datasets.
- **Testing:** Add unit tests that assert no leakage and validate expected aggregations on known inputs.

Next: run `02-Training-MLflow-Tracking-Model-Registry.ipynb` to train a model using `demo.enriched_training_set`.